# The Transformer Architecture

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/03-transformer-architecture

A from-scratch, runnable implementation of the concepts in the lesson.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## A full Transformer block in numpy

Assemble the pieces: multi-head self-attention + a position-wise feed-forward net, each wrapped in a **residual** connection and **LayerNorm** (pre-LN style).

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True); e = np.exp(x)
    return e/e.sum(axis=axis, keepdims=True)

def layernorm(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

def mha(X, Wq, Wk, Wv, Wo, h, mask=None):
    T, d = X.shape; dk = d//h
    Q=(X@Wq).reshape(T,h,dk).transpose(1,0,2); K=(X@Wk).reshape(T,h,dk).transpose(1,0,2)
    V=(X@Wv).reshape(T,h,dk).transpose(1,0,2)
    s=Q@K.transpose(0,2,1)/np.sqrt(dk)
    if mask is not None: s=np.where(mask, s, -1e9)
    ctx=(softmax(s)@V).transpose(1,0,2).reshape(T,d)
    return ctx@Wo

def relu(x): return np.maximum(0, x)

class TransformerBlock:
    def __init__(self, d, h, d_ff, seed=0):
        rng=np.random.RandomState(seed); self.h=h
        self.Wq,self.Wk,self.Wv,self.Wo = (rng.randn(d,d)*0.1 for _ in range(4))
        self.W1=rng.randn(d,d_ff)*0.1; self.b1=np.zeros(d_ff)
        self.W2=rng.randn(d_ff,d)*0.1; self.b2=np.zeros(d)
    def __call__(self, X, mask=None):
        X = X + mha(layernorm(X), self.Wq,self.Wk,self.Wv,self.Wo, self.h, mask)  # residual
        ff = relu(layernorm(X)@self.W1 + self.b1)@self.W2 + self.b2
        return X + ff                                                            # residual

d_model, n_heads, d_ff, T = 32, 4, 128, 7
X = np.random.RandomState(1).randn(T, d_model)
block = TransformerBlock(d_model, n_heads, d_ff)
print('block output:', block(X).shape, '(same shape in, same shape out)')

## LayerNorm, worked by hand

LayerNorm standardizes across the **feature** dimension of each token (not the batch), then applies a learned scale $\gamma$ and shift $\beta$:

$$\mu = \tfrac{1}{d}\sum_i x_i,\quad \sigma^2 = \tfrac{1}{d}\sum_i (x_i-\mu)^2,\quad y = \gamma\,\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta.$$

We reproduce the lesson's tiny example $x=[2,4,4,4,5,5,7,9]$ in **pure stdlib** so the output is exact and deterministic. Expect $\mu=5$, $\sigma^2=4$, $\sigma=2$, normalized $=[-1.5,-0.5,-0.5,-0.5,0,0,1,2]$ (mean 0, var 1), and with $\gamma=2,\beta=1$: $[-2,0,0,0,1,1,3,5]$.

In [ ]:
import math

# --- LayerNorm worked by hand (pure stdlib, deterministic) ---
x = [2, 4, 4, 4, 5, 5, 7, 9]               # one token's d=8 feature vector
d = len(x)
mu = sum(x) / d                            # mean across features
var = sum((xi - mu) ** 2 for xi in x) / d  # population variance (as in LayerNorm)
std = math.sqrt(var)
norm = [(xi - mu) / std for xi in x]       # eps negligible here
gamma, beta = 2.0, 1.0                     # learned affine params (broadcast)
y = [gamma * n + beta for n in norm]

print("mu   =", mu)
print("var  =", var, " std =", std)
print("norm =", [round(v, 4) for v in norm])
print("mean(norm) =", round(sum(norm) / d, 12),
      " var(norm) =", round(sum(v ** 2 for v in norm) / d, 12))
print("y = gamma*norm + beta =", [round(v, 4) for v in y])

assert mu == 5.0 and var == 4.0 and std == 2.0
assert [round(v, 4) for v in norm] == [-1.5, -0.5, -0.5, -0.5, 0.0, 0.0, 1.0, 2.0]
assert [round(v, 4) for v in y] == [-2.0, 0.0, 0.0, 0.0, 1.0, 1.0, 3.0, 5.0]
print("LayerNorm hand-check: PASS")

## Counting an encoder layer's parameters

One encoder layer = attention + FFN + two LayerNorms (biases ignored):

- **Attention** $W_Q, W_K, W_V, W_O$, each $d_{\text{model}}\times d_{\text{model}}$ → $4\,d_{\text{model}}^2$
- **FFN** $W_1\ (d_{\text{model}}\times d_{ff})$ + $W_2\ (d_{ff}\times d_{\text{model}})$ → $2\,d_{\text{model}}\,d_{ff}$
- **2 LayerNorms** ($\gamma,\beta$ each length $d_{\text{model}}$) → $4\,d_{\text{model}}$

For $d_{\text{model}}=512,\ d_{ff}=2048,\ h=8$ this is $1{,}048{,}576 + 2{,}097{,}152 + 2{,}048 = 3{,}147{,}776 \approx 3.15$M. The head count $h$ does not change the total (per-head projections concatenate back to $d\times d$). FFN is $\approx 66.6\%$ of the parameters.

In [ ]:
# --- Encoder-layer parameter count (pure stdlib, deterministic) ---
d_model, d_ff, h = 512, 2048, 8

attn = 4 * d_model ** 2       # Wq, Wk, Wv, Wo each d x d
ffn  = 2 * d_model * d_ff     # W1 (d x d_ff) + W2 (d_ff x d)
ln   = 2 * (2 * d_model)      # two LayerNorms, gamma+beta each length d
total = attn + ffn + ln

print(f"attention 4*d^2  = {attn:,}")
print(f"FFN 2*d*d_ff     = {ffn:,}")
print(f"2 LayerNorms 4*d = {ln:,}")
print(f"total per layer  = {total:,}  (~{total/1e6:.2f}M)")
print(f"FFN share        = {ffn/total*100:.1f}%   attention share = {attn/total*100:.1f}%")
print(f"6-layer encoder  ~ {6*total/1e6:.1f}M params (before embeddings)")

assert attn == 1_048_576 and ffn == 2_097_152 and ln == 2_048
assert total == 3_147_776
print("param-count hand-check: PASS")

## Stacking blocks = a deep Transformer

Real models stack dozens of identical blocks. Residuals keep activations stable as depth grows — we track the activation norm through the stack.

In [ ]:
blocks = [TransformerBlock(d_model, n_heads, d_ff, seed=i) for i in range(12)]
norms = [np.linalg.norm(X)]
h = X
for blk in blocks:
    h = blk(h); norms.append(np.linalg.norm(h))
plt.plot(norms, 'o-', color='#6366f1'); plt.xlabel('block depth'); plt.ylabel('||activations||')
plt.title('Residual connections keep a 12-block stack stable'); plt.show()

## Decoder-only forward pass (GPT-style)

Add token embeddings + positional encoding, run causal blocks, project to vocab logits, softmax → next-token distribution. Here, untrained, just to show the data flow and shapes.

In [ ]:
vocab, d_model, T = 50, 32, 7
rng = np.random.RandomState(2)
tok_emb = rng.randn(vocab, d_model)*0.1
def pos_enc(T, d):
    p=np.arange(T)[:,None]; i=np.arange(d)[None,:]; a=p/np.power(10000,(2*(i//2))/d)
    pe=np.zeros((T,d)); pe[:,0::2]=np.sin(a[:,0::2]); pe[:,1::2]=np.cos(a[:,1::2]); return pe

tokens = rng.randint(0, vocab, size=T)
Xh = tok_emb[tokens] + pos_enc(T, d_model)
mask = np.tril(np.ones((T, T))).astype(bool)
for blk in blocks:
    Xh = blk(Xh, mask=mask)
W_out = rng.randn(d_model, vocab)*0.1
logits = layernorm(Xh) @ W_out
probs = softmax(logits)
print('next-token distribution shape:', probs.shape)
print('predicted next token after position', T-1, '->', int(probs[-1].argmax()))

## Key takeaways

- A block alternates **attention** (mix tokens) and a **feed-forward** net (per-token compute).
- **Residuals + LayerNorm** are what make deep stacks trainable — activation norm stays bounded.
- A causal mask turns the encoder block into a GPT-style **decoder**.
- The full forward pass is: embed + position → N blocks → project → softmax over the vocab.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — LayerNorm

Normalize **each row** (each token's features) to zero mean and unit variance, then apply a learnable affine:

$$\text{LN}(x) = \gamma \, \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

The last check is the property that makes training stable: LayerNorm's output is **invariant to scaling and shifting** of its input row.

In [ ]:
def layer_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    """LayerNorm over the last axis."""
    x = np.asarray(x, dtype=float)

    # TODO(you): per-row mean and variance (keepdims=True)
    mu = ...
    var = ...

    # TODO(you): normalize, then gamma * . + beta
    return ...

In [ ]:
# Checks — run me
x = np.array([[1.0, 2.0, 3.0, 4.0], [10.0, 10.0, 20.0, 20.0]])

out = layer_norm(x)
assert np.allclose(out.mean(axis=-1), 0, atol=1e-9), "each row is centered"
assert np.allclose(out.std(axis=-1), 1, atol=1e-2), "each row has ~unit std"

assert np.allclose(layer_norm(x, gamma=2.0, beta=3.0), 2.0 * out + 3.0), "gamma/beta are a learnable affine on top"
assert np.allclose(layer_norm(x * 100 + 7)[0], out[0], atol=1e-3), "invariant to scale and shift of the input row"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def layer_norm(x, gamma=1.0, beta=0.0, eps=1e-5):
    x = np.asarray(x, dtype=float)
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta
```

</details>

### Exercise 2 — An encoder layer's parameter budget

Add up one encoder layer: multi-head attention ($4(d^2+d)$), the two-layer FFN ($d \to d_{ff} \to d$, with biases), and two LayerNorms ($2d$ each). For $d = 512, d_{ff} = 2048$ that's **3,152,384** — and the punchline of the last check: the FFN, not attention, holds most of the parameters.

In [ ]:
def encoder_layer_params(d_model, d_ff):
    """Total parameters of one encoder layer."""
    # TODO(you): attention: 4 * (d^2 + d)
    attn = ...

    # TODO(you): FFN: (d*d_ff + d_ff) + (d_ff*d + d)
    ffn = ...

    # TODO(you): two LayerNorms, each with gamma and beta of size d
    ln = ...

    return attn + ffn + ln

In [ ]:
# Checks — run me
assert encoder_layer_params(512, 2048) == 1050624 + 2099712 + 2048, "attention + FFN + 2 LayerNorms"
assert encoder_layer_params(512, 2048) == 3152384, "~3.15M per layer — x6 layers ~ 19M, the original encoder"

ffn_share = (512 * 2048 + 2048 + 2048 * 512 + 512) / encoder_layer_params(512, 2048)
assert ffn_share > 0.6, "the FFN, not attention, holds most of the parameters"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def encoder_layer_params(d_model, d_ff):
    attn = 4 * (d_model * d_model + d_model)
    ffn = d_model * d_ff + d_ff + d_ff * d_model + d_model
    ln = 2 * (2 * d_model)
    return attn + ffn + ln
```

</details>